In [1]:
# 1. Libraries

import pandas as pd
import matplotlib.pyplot as plt
import dash
from dash import dcc, html, Input, Output, State
import dash_bootstrap_components as dbc
import plotly.express as px

In [2]:
# 2. Import File
path_to_file = "https://raw.githubusercontent.com/cesarlarasantana/VDS_2526_G04_Football/refs/heads/main/Datasets"

df_country = pd.read_csv(path_to_file+"/Country.csv")
df_match_goals = pd.read_csv(path_to_file+"/Match_Goals.csv")
df_match_shots_on = pd.read_csv(path_to_file+"/Match_Shots_On.csv")
df_match_fouls = pd.read_csv(path_to_file+"/Match_Fouls_Committed.csv")
df_match_cards = pd.read_csv(path_to_file+"/Match_Cards.csv")
df_team = pd.read_csv(path_to_file+"/Team.csv")
df_match = pd.read_csv(path_to_file+"/Match.csv")
df_player = pd.read_csv(path_to_file+"/Player.csv")
df_player_att = pd.read_csv(path_to_file+"/Player_Attributes.csv", sep= ';')
df_position_ref = pd.read_csv(path_to_file+"/PositionReference.csv")


C:\Users\pci\AppData\Local\Temp\ipykernel_1452\4289516213.py:7: DtypeWarning: Columns (0: player1) have mixed types. Specify dtype option on import or set low_memory=False.
  df_match_fouls = pd.read_csv(path_to_file+"/Match_Fouls_Committed.csv")


In [3]:
# 3. Data Preparation

#3.1 GOALS
### Only valid goals: 1) n: normal, 2) p: penalty

df_match_goals_valid =  df_match_goals[(df_match_goals['goal_type'] == "n") | (df_match_goals['goal_type'] == "p")].reset_index()
agg_df_match_goals = df_match_goals_valid.groupby(['match_id', 'team']).agg({'goal_type':'count'}).reset_index()

#3.2 FOULS
agg_df_match_fouls = df_match_fouls.groupby(['match_id', 'team']).agg({'elapsed':'count'}).reset_index()

#3.3 CARDS
agg_df_match_cards = df_match_cards.groupby(['match_id', 'team']).agg({'elapsed':'count'}).reset_index()

#3.4 SHOTS-ON

agg_df_match_shots_on = df_match_shots_on.groupby(['match_id', 'team']).agg({'elapsed':'count'}).reset_index()

#3.5 MATCH OUTCOME

df_match['winning_team_id'] = 9999999
df_match['winning_team_goals'] = 0
df_match['lossing_team_id'] = 9999999
df_match['lossing_team_goals'] = 0
df_match['draw_team1_id'] = 0
df_match['draw_team2_id'] = 0
df_match['draw_team1_goals'] = 0
df_match['draw_team2_goals'] = 0

for i in range(0, len(df_match)):
    if(df_match.iloc[i]['home_team_goal'] > df_match.iloc[i]['away_team_goal']):
        df_match.at[i, 'winning_team_id'] = df_match.iloc[i]['home_team_api_id']
        df_match.at[i, 'winning_team_goals'] = df_match.iloc[i]['home_team_goal']

        df_match.at[i, 'lossing_team_id'] = df_match.iloc[i]['away_team_api_id']
        df_match.at[i, 'lossing_team_goals'] = df_match.iloc[i]['away_team_goal']

    elif(df_match.iloc[i]['home_team_goal'] < df_match.iloc[i]['away_team_goal']):
        df_match.at[i, 'winning_team_id'] = df_match.iloc[i]['away_team_api_id']
        df_match.at[i, 'winning_team_goals'] = df_match.iloc[i]['away_team_goal']

        df_match.at[i, 'lossing_team_id'] = df_match.iloc[i]['home_team_api_id']
        df_match.at[i, 'lossing_team_goals'] = df_match.iloc[i]['home_team_goal']
    else:
        df_match.at[i, 'draw_team1_id'] = df_match.iloc[i]['home_team_api_id']
        df_match.at[i, 'draw_team2_id'] = df_match.iloc[i]['away_team_api_id']

        df_match.at[i, 'draw_team1_goals'] = df_match.iloc[i]['home_team_goal']
        df_match.at[i, 'draw_team2_goals'] = df_match.iloc[i]['away_team_goal']


In [4]:

# 3.6 Analysis Per Game 

## Winning Result per Game, Season and Team
agg_df_match_team_win = df_match.groupby(['match_api_id', 'winning_team_id', 'season']).agg({'stage':'count',
                                                                             'winning_team_goals': 'sum'}).reset_index()
agg_df_match_team_win = agg_df_match_team_win.rename({'winning_team_id' : 'team',
                                                      'stage': 'num',
                                                      'winning_team_goals': 'sum_goals'
                                                      }, axis=1)
agg_df_match_team_win['game_result'] = '1_Win'
agg_df_match_team_win = agg_df_match_team_win[agg_df_match_team_win['team'] != 9999999]

## Lossing Result per Game, Season and Team
agg_df_match_team_los = df_match.groupby(['match_api_id', 'lossing_team_id', 'season']).agg({'stage':'count',
                                                                             'lossing_team_goals': 'sum'}).reset_index()
agg_df_match_team_los = agg_df_match_team_los.rename({'lossing_team_id' : 'team',
                                                      'stage': 'num',
                                                      'lossing_team_goals': 'sum_goals'
                                                      }, axis=1)
agg_df_match_team_los['game_result'] = '2_Loss'
agg_df_match_team_los = agg_df_match_team_los[agg_df_match_team_los['team'] != 9999999]

## Draw Result Game, Season and Team - DRAW 1

agg_df_match_team_draw_1 = df_match.groupby(['match_api_id', 'draw_team1_id', 'season']).agg({'stage':'count',
                                                                             'draw_team1_goals': 'sum'}).reset_index()
agg_df_match_team_draw_1 = agg_df_match_team_draw_1.rename({'draw_team1_id' : 'team',
                                                      'stage': 'num',
                                                      'draw_team1_goals': 'sum_goals'
                                                      }, axis=1)
agg_df_match_team_draw_1['game_result'] = '3_Draw'
agg_df_match_team_draw_1 = agg_df_match_team_draw_1[agg_df_match_team_draw_1['team'] != 0]


## Draw Result Game, Season and Team - DRAW 2

agg_df_match_team_draw_2 = df_match.groupby(['match_api_id', 'draw_team2_id', 'season']).agg({'stage':'count',
                                                                             'draw_team2_goals': 'sum'}).reset_index()
agg_df_match_team_draw_2 = agg_df_match_team_draw_2.rename({'draw_team2_id' : 'team',
                                                      'stage': 'num',
                                                      'draw_team2_goals': 'sum_goals'
                                                      }, axis=1)
agg_df_match_team_draw_2['game_result'] = '3_Draw'
agg_df_match_team_draw_2 = agg_df_match_team_draw_2[agg_df_match_team_draw_2['team'] != 0]

agg_df_match_team = pd.concat([agg_df_match_team_win, agg_df_match_team_los, agg_df_match_team_draw_1, agg_df_match_team_draw_2]).reset_index()

agg_df_match_team = agg_df_match_team.sort_values(by = ['season','match_api_id', 'team']).reset_index()


In [5]:
# New Data Preparation
## For every Team, each season, count the number of victories, losses and draws

# Win Data
agg_df_match_team_w = agg_df_match_team[agg_df_match_team['game_result'] == '1_Win']
agg_team_wins = agg_df_match_team_w.groupby(['season', 'team']).agg({'sum_goals': 'sum',
                                                                     'num': 'count'}).reset_index()
agg_team_wins = agg_team_wins.rename({'sum_goals': 'sum_goals_win',
                                      'num' : 'num_games_wins'}, axis=1)

# Loss Data
agg_df_match_team_l = agg_df_match_team[agg_df_match_team['game_result'] == '2_Loss']
agg_team_losses = agg_df_match_team_l.groupby(['season', 'team']).agg({'sum_goals': 'sum',
                                                                     'num': 'count'}).reset_index()

agg_team_losses = agg_team_losses.rename({'sum_goals': 'sum_goals_losses',
                                         'num' : 'num_games_losses'}, axis=1)

# Draw Data
agg_df_match_team_d = agg_df_match_team[agg_df_match_team['game_result'] == '3_Draw']
agg_team_draws = agg_df_match_team_d.groupby(['season', 'team']).agg({'sum_goals': 'sum',
                                                                     'num': 'count'}).reset_index()

agg_team_draws = agg_team_draws.rename({'sum_goals': 'sum_goals_draws',
                                        'num' : 'num_games_draws'}, axis=1)

## Merging The three Data Frames

df_1 = agg_team_wins.merge(agg_team_losses, on=['season', 'team'], how = 'outer').reset_index()
df_1_team_summary = df_1.merge(agg_team_draws, on=['season', 'team'], how = 'outer').reset_index()
df_1_team_summary['total_games'] = df_1_team_summary['num_games_wins'] + df_1_team_summary['num_games_losses'] + df_1_team_summary['num_games_draws']
df_1_team_summary['prct_win'] = round((df_1_team_summary['num_games_wins']/df_1_team_summary['total_games'])*100, 2)

df_1_team_summary = df_1_team_summary.rename({'team': 'team_api_id'}, axis=1)

## Identifying which Tier does the Team belongs

df_1_team_summary['tier_group'] = ""

for i in range(0, len(df_1_team_summary)):
    if(df_1_team_summary.iloc[i]['prct_win'] > 90):
        df_1_team_summary.at[i, 'tier_group'] = "1_Top_10%"
    elif(df_1_team_summary.iloc[i]['prct_win'] > 80):
        df_1_team_summary.at[i, 'tier_group'] = "2_Top_20%"
    elif(df_1_team_summary.iloc[i]['prct_win'] > 70):
        df_1_team_summary.at[i, 'tier_group'] = "3_Top_30%"
    elif(df_1_team_summary.iloc[i]['prct_win'] > 60):
        df_1_team_summary.at[i, 'tier_group'] = "4_Top_40%"
    elif(df_1_team_summary.iloc[i]['prct_win'] > 50):
        df_1_team_summary.at[i, 'tier_group'] = "5_Top_50%"
    else:
        df_1_team_summary.at[i, 'tier_group'] = "6_Below_50%"

## Include the Team Name

df_1_team_summary = df_1_team_summary.merge(df_team[['team_api_id', 'team_long_name']], on='team_api_id', how='left')
df_1_team_summary = df_1_team_summary.drop({'index', 'level_0'}, axis=1)

In [6]:
## Visualization as Dash App

app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])
app.layout = dbc.Container([
    html.H1("CE_TM_04 - Team Wining Evolution", className="my-3"),
    html.H4("Select the Team to Analyze", className="my-3"),
    dbc.Row([
        dbc.Row([
            dcc.Dropdown(id='category-dropdown', options=df_1_team_summary['team_long_name'].unique(), clearable=False),
            dbc.Button("Update Graph", id="update-btn", color="primary", className="mt-2")
        ]),
        dbc.Row([dcc.Graph(id='main-graph')])
    ])
])

# 3. Callback to Update Graph
@app.callback(
    Output('main-graph', 'figure'),
    Input('update-btn', 'n_clicks'),
    State('category-dropdown', 'value'),
    prevent_initial_call=False
)

def update_graph(n_clicks, selected_category):
    row_to_higlight = df_1_team_summary.loc[df_1_team_summary['team_long_name'] == selected_category,]
    row_top_10 = df_1_team_summary.loc[df_1_team_summary['tier_group'] == '1_Top_10%',]
    row_top_20 = df_1_team_summary.loc[df_1_team_summary['tier_group'] == '2_Top_20%',]
    row_top_30 = df_1_team_summary.loc[df_1_team_summary['tier_group'] == '3_Top_30%',]

    plt.figure(figsize=(16, 40))

    fig = px.box(df_1_team_summary,
                    x='season', y='prct_win')
    
    fig.update_traces(line_color='black', fillcolor="#d3d3d3")

    fig.add_scatter(
        x=row_to_higlight['season'], 
        y=row_to_higlight['prct_win'],
        mode='markers',
        marker=dict(color='red', size=14, symbol='diamond', line=dict(width=2, color='red')),
        name=selected_category,
        # Ensure it aligns with the correct subgroup
        # Use 'offsetgroup' if you need precise alignment within grouped boxes
    )

    fig.add_scatter(
        x=row_top_10['season'], 
        y=row_top_10['prct_win'],
        mode='markers',
        marker=dict(color="#071cd4", size=10, symbol='circle', line=dict(width=2, color='darkblue')),
        name='team_long_name',
        # Ensure it aligns with the correct subgroup
        # Use 'offsetgroup' if you need precise alignment within grouped boxes
    )

    fig.add_scatter(
        x=row_top_20['season'], 
        y=row_top_20['prct_win'],
        mode='markers',
        marker=dict(color="#4c5ad8", size=9, symbol='circle', line=dict(width=2, color='blue')),
        name='2_Top_20%',
        # Ensure it aligns with the correct subgroup
        # Use 'offsetgroup' if you need precise alignment within grouped boxes
    )

    fig.add_scatter(
        x=row_top_30['season'], 
        y=row_top_30['prct_win'],
        mode='markers',
        marker=dict(color="#3fc2da", size=9, symbol='square', line=dict(width=2, color='lightblue')),
        name='3_Top_30%',
        # Ensure it aligns with the correct subgroup
        # Use 'offsetgroup' if you need precise alignment within grouped boxes
    )

    fig.update_layout(xaxis_title="Season", yaxis_title="Percentage of Games Won in each Season")

    
    return fig

if __name__ == '__main__':
    app.run(debug=True, port=8010)